<a href="https://colab.research.google.com/github/Amyerm/ClassFiles/blob/main/Actividad_Ejercicio2_OpenMeteo_Sesion8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad — Comparar el clima entre varias ciudades con Open-Meteo (Sesión 8)

En el Bloque 2 consultaste el pronóstico por hora de Ciudad Juárez con Open-Meteo. En esta actividad usas
la misma API, pero de otra forma: en vez de un pronóstico por hora de una sola ciudad, se consulta el
**clima actual** de **varias ciudades distintas** y se construye una tabla comparativa.

**Endpoint:** `https://api.open-meteo.com/v1/forecast`
**Parámetro clave:** `current_weather=true` — regresa un solo valor actual (no una lista por hora) dentro
de `datos["current_weather"]`, con `temperature`, `windspeed`, `weathercode` y `time`.

**Instrucciones:**

1. Define una lista `ciudades` con al menos tres ciudades (nombre, latitud, longitud), distintas a
   Ciudad Juárez.
2. Para cada ciudad, consulta el endpoint con `current_weather=true`, usando `timeout` y manejo de
   errores como en el Bloque 2.
3. De cada respuesta, extrae la temperatura y la velocidad del viento actuales.
4. Estructura la información como una lista de diccionarios llamada `info_clima`.
5. Convierte `info_clima` en un DataFrame llamado `df_clima_ciudades`.
6. Verifica el resultado.

**Paso 1 — Define las ciudades a consultar.**

In [5]:
ciudades = [
    {"nombre": "Chihuahua", "latitud": 28.63, "longitud": -106.09},
    {"nombre": "Monterrey", "latitud": 25.69, "longitud": -100.32},
    {"nombre": "Guadalajara", "latitud": 20.66, "longitud": -103.35},
]  # puedes cambiar o agregar ciudades

**Pasos 2 a 4 — Consultar cada ciudad y estructurar la información.**

In [6]:
import requests

info_clima = []

for ciudad in ciudades:
    endpoint = "https://api.open-meteo.com/v1/forecast"
    parametros = {
        "latitude": ciudad["latitud"],
        "longitude": ciudad["longitud"],
        "current_weather": True,
    }

    try:
        respuesta = requests.get(
            endpoint,
            params=parametros,
            timeout=30
        )

        respuesta.raise_for_status()

        datos = respuesta.json()
        clima_actual = datos["current_weather"]

        info_clima.append({
            "ciudad": ciudad["nombre"],
            "temperatura": clima_actual["temperature"],
            "viento": clima_actual["windspeed"],
        })

    except requests.RequestException as error:
        print(f"No fue posible consultar {ciudad['nombre']}:", error)

info_clima

[{'ciudad': 'Chihuahua', 'temperatura': 27.5, 'viento': 32.0},
 {'ciudad': 'Monterrey', 'temperatura': 29.0, 'viento': 27.2},
 {'ciudad': 'Guadalajara', 'temperatura': 19.4, 'viento': 0.6}]

**Paso 5 — Construir el DataFrame.**

In [7]:
import pandas as pd

df_clima_ciudades = pd.DataFrame(info_clima)

df_clima_ciudades

,ciudad,temperatura,viento
0,Chihuahua,27.5,32.0
1,Monterrey,29.0,27.2
2,Guadalajara,19.4,0.6


**Paso 6 — Verificar el resultado.**

In [8]:
print("Dimensiones:", df_clima_ciudades.shape)

print("Columnas:", df_clima_ciudades.columns.tolist())

print()

print("Valores faltantes por columna:")
print(df_clima_ciudades.isna().sum())

Dimensiones: (3, 3)
Columnas: ['ciudad', 'temperatura', 'viento']

Valores faltantes por columna:
ciudad         0
temperatura    0
viento         0
dtype: int64
